In [4]:
# Cell 2: Mount Google Drive and set up Git user
import os
from google.colab import drive
drive.mount('/content/drive')

# Configure Git (replace with your own name/email)
!git config --global user.name "Your Name"
!git config --global user.email "your.email@example.com"

# Clone the repository (change URL if needed)
!git clone https://github.com/inshiright/chestxray-classification.git
%cd chestxray-classification

print(f"Current working directory: {os.getcwd()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cloning into 'chestxray-classification'...
remote: Enumerating objects: 5772, done.
remote: Counting objects: 100% (178/178), done.
remote: Compressing objects: 100% (135/135), done.
remote: Total 5772 (delta 82), reused 99 (delta 26), pack-reused 5594 (from 2)
Receiving objects: 100% (5772/5772), 25.31 MiB | 20.57 MiB/s, done.
Resolving deltas: 100% (1729/1729), done.
/content/chestxray-classification
Current working directory: /content/chestxray-classification


In [5]:
# Cell 3: Install Python dependencies
!pip install -r requirements.txt
!pip install transformers shap lime pytorch-gradcam

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 40.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.9 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opencv-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 11.3 MB/s eta 0:00:00
  Created wheel for grad-cam: filename=grad_cam-1.5.5-py3-none-any.whl size=44286 sha256=7d59b6b2fb6247c7d60edcb5c5f3fe40e94cc3fa7e52ac625b594cdd03f68fc2
  Stored in directory: /root/.cache/pip/wheels/fb/3b/09/2afc520f3d69bc26ae6bd87416759c820a3f7d05c1a077bbf6
Successfully built grad-cam
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
    

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 14.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 60.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 26.1 MB/s eta 0:00:00
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283834 sha256=21b67033838077703a7ce19ceb6a1843b4fb5aac63c1211495c5cc50dcf6fed2
  Stored in directory: /root/.cache/pip/wheels/e7/5d/0e/4b4fff9a47468fed5633211fb3b76d1db43fe806a17fb7486a
  Created wheel for pytorch-gradcam: filename=pytorch_gradcam-0.2.1-py3-none-any.whl size=5249 sha256=5067ef7fd4c6f49d1b8045d97287cb4bb150dde59e543a5033d39b6c7d01305e
  Stored in directory: /root/.cache/pip/wheels/b3/37/60/ab0eb271797fc9bf2a8b6edd8e550f2ee0857031e4cc0e2509
Successfully built lime pytorch-gradcam
  Attempting uninstall: numpy

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:

import os
import shutil

# 2. Define source and destination
SOURCE_ZIP = "/content/drive/Shareddrives/adl_cv_data/data.zip"

DESTINATION_DIR = "/content/dataset"

# 3. Create the local directory
os.makedirs(DESTINATION_DIR, exist_ok=True)

print("Starting Transfer from Drive to Local SSD (this is fast)...")

# 4. Copy the zip file
!cp "{SOURCE_ZIP}" /content/data_local.zip
# !gdown "1bO0S2WeRnu1l25wDbhFmgrP9yUK_g93J" -O /content/data_local.zip

print("Transfer complete. Starting extraction (this may take 15-20 mins)...")

# 5. Unzip the file into the local folder
!unzip -q /content/data_local.zip -d "{DESTINATION_DIR}"

# 6. Clean up the local zip
os.remove("/content/data_local.zip")

print(f"Done! Your images are ready at {DESTINATION_DIR}")

Starting Transfer from Drive to Local SSD (this is fast)...
Transfer complete. Starting extraction (this may take 15-20 mins)...
Done! Your images are ready at /content/dataset


In [7]:
# Cell 5: Update config.py to point to the dataset location
# (The repository's config.py may exist, but we override it to ensure correct paths)
%%writefile src/config.py
import os

ROOT_DIR = os.path.abspath(os.path.join(os.path.dirname(__file__), ".."))

# Pointing to the extracted dataset location
DATASET_DIR = "/content/dataset"
CSV_PATH = os.path.join(DATASET_DIR, "Data_Entry_2017.csv")

# Hyperparameters
IMAGE_SIZE = 224
NUM_CLASSES = 14
BATCH_SIZE = 16
EPOCHS = 30
LR = 1e-4

# Active Model
MODEL_NAME = "raddino"

# Checkpoint Settings
CHECKPOINT_DIR = os.path.join(ROOT_DIR, "checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
RESUME_CHECKPOINT_PATH = None

Overwriting src/config.py


In [8]:

!python src/main.py

# To resume training from a checkpoint (if available):
# !python src/resume_train.py checkpoints/raddino_best_model.pth

Traceback (most recent call last):
  File "/content/chestxray-classification/src/main.py", line 220, in <module>
    main()
  File "/content/chestxray-classification/src/main.py", line 29, in main
    df = build_dataframe_with_paths(csv_path=CSV_PATH, dataset_root=IMAGE_DIR)
                                                                    ^^^^^^^^^
NameError: name 'IMAGE_DIR' is not defined. Did you mean: 'IMAGE_SIZE'?
